# Análisis Exploratorio de Datos — Estudio PENSER Egresados USTA
**Proyecto:** Consultorio de Estadística y Ciencia de Datos  
**Equipo:** Yeimy Alarcón · Karen Suarez · Maria José Galindo  
**Semestre:** 2025-2

Este notebook presenta el análisis exploratorio de la base de datos del Estudio PENSER,
que recoge percepciones, competencias y trayectorias de graduados de la Universidad Santo Tomás.

---

## 1. Configuración e Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Estilo
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')
COLORES = ['#2C5F2D', '#97BC62', '#F96167', '#065A82', '#F9E795']

print('Librerías cargadas correctamente.')

## 2. Carga de Datos

In [ ]:
# Base cruda (antes de limpieza)
df_raw = pd.read_excel('data/raw/ESTUDIO_DE_PERCEPCION_EGRESADOS.xlsx')

# Base procesada (después del pipeline completo)
df = pd.read_parquet('data/processed/base_procesada.parquet')

print('═' * 50)
print(f'  Base cruda    : {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
print(f'  Base procesada: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'  Registros eliminados en limpieza: {df_raw.shape[0] - df.shape[0]}')
print('═' * 50)

## 3. Calidad de Datos — Valores Nulos

In [ ]:
# Porcentaje de nulos por columna en la base cruda
nulos_pct = (df_raw.isnull().mean() * 100).sort_values(ascending=False)
nulos_pct = nulos_pct[nulos_pct > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de nulos
bins = [0, 10, 30, 50, 70, 90, 100]
labels_bins = ['0-10%', '10-30%', '30-50%', '50-70%', '70-90%', '90-100%']
cat = pd.cut(nulos_pct, bins=bins, labels=labels_bins)
cat.value_counts().sort_index().plot(kind='bar', ax=axes[0], color=COLORES[1], edgecolor='white')
axes[0].set_title('Distribución de columnas por % de nulos', fontweight='bold')
axes[0].set_xlabel('Rango de nulos')
axes[0].set_ylabel('Número de columnas')
axes[0].tick_params(axis='x', rotation=30)

# Top 15 columnas con más nulos
top15 = nulos_pct.head(15)
top15.plot(kind='barh', ax=axes[1], color=COLORES[2], edgecolor='white')
axes[1].set_title('Top 15 columnas con más nulos', fontweight='bold')
axes[1].set_xlabel('% de valores nulos')
axes[1].axvline(x=95, color='red', linestyle='--', alpha=0.7, label='Umbral 95%')
axes[1].legend()
axes[1].set_yticklabels([t[:40] + '...' if len(t) > 40 else t for t in top15.index], fontsize=8)

plt.tight_layout()
plt.suptitle('Análisis de Valores Nulos — Base Cruda PENSER', y=1.02, fontsize=13, fontweight='bold')
plt.show()

print(f'Columnas con >95% nulos (eliminadas): {(nulos_pct > 95).sum()}')
print(f'Columnas con <20% nulos (conservadas): {(nulos_pct < 20).sum()}')

## 4. Perfil Sociodemográfico

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Género
genero_cols = [c for c in df.columns if c.startswith('genero_')]
if genero_cols:
    genero = df[genero_cols].sum().sort_values(ascending=False)
    genero.index = [c.replace('genero_', '') for c in genero.index]
    genero.plot(kind='bar', ax=axes[0], color=COLORES, edgecolor='white')
    axes[0].set_title('Distribución por Género', fontweight='bold')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Número de graduados')
    axes[0].tick_params(axis='x', rotation=30)
    for i, v in enumerate(genero):
        axes[0].text(i, v + 5, f'{v/len(df)*100:.1f}%', ha='center', fontsize=9)

# Sede
sede_cols = [c for c in df.columns if c.startswith('sede_')]
if sede_cols:
    sede = df[sede_cols].sum().sort_values(ascending=False)
    sede.index = [c.replace('sede_', '') for c in sede.index]
    sede.plot(kind='bar', ax=axes[1], color=COLORES[3], edgecolor='white')
    axes[1].set_title('Distribución por Sede', fontweight='bold')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Número de graduados')
    axes[1].tick_params(axis='x', rotation=30)
    for i, v in enumerate(sede):
        axes[1].text(i, v + 5, str(v), ha='center', fontsize=9)

# Nivel de formación
if 'nivel_formacion' in df.columns:
    nivel_map = {1: 'Pregrado', 2: 'Especialización', 3: 'Maestría', 4: 'Doctorado'}
    nivel = pd.to_numeric(df['nivel_formacion'], errors='coerce').map(nivel_map).value_counts()
    nivel.plot(kind='bar', ax=axes[2], color=COLORES[1], edgecolor='white')
    axes[2].set_title('Nivel de Formación', fontweight='bold')
    axes[2].set_xlabel('')
    axes[2].set_ylabel('Número de graduados')
    axes[2].tick_params(axis='x', rotation=30)
    for i, v in enumerate(nivel):
        axes[2].text(i, v + 5, f'{v/len(df)*100:.1f}%', ha='center', fontsize=9)

plt.suptitle('Perfil Sociodemográfico de Graduados USTA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Distribución de Competencias (Escala Likert 1–5)

In [ ]:
COLS_COMPETENCIAS = [
    'com_escrita', 'com_oral', 'pensamiento_critico', 'metodos_cuantitativos',
    'metodos_cualitativos', 'lectura_academica', 'argumentacion', 'segunda_lengua',
    'creatividad', 'resolucion_conflictos', 'liderazgo', 'toma_decisiones',
    'resolucion_problemas', 'investigacion', 'herramientas_informaticas',
    'contextos_multiculturales', 'insercion_laboral', 'herramientas_modernas',
    'gestion_informacion', 'trabajo_equipo', 'aprendizaje_autonomo',
    'conocimientos_multidisciplinares', 'etica'
]

cols = [c for c in COLS_COMPETENCIAS if c in df.columns]
medias = df[cols].apply(lambda x: pd.to_numeric(x, errors='coerce')).mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Medias ordenadas
colores_barra = ['#F96167' if v < 3.5 else '#97BC62' if v >= 4.0 else '#F9E795' for v in medias]
medias.plot(kind='barh', ax=axes[0], color=colores_barra, edgecolor='white')
axes[0].set_title('Media por Competencia', fontweight='bold')
axes[0].set_xlabel('Media (escala 1–5)')
axes[0].axvline(x=medias.mean(), color='gray', linestyle='--', alpha=0.7, label=f'Media global: {medias.mean():.2f}')
axes[0].axvline(x=3.0, color='red', linestyle=':', alpha=0.5, label='Umbral 3.0')
axes[0].legend(fontsize=9)
for i, v in enumerate(medias):
    axes[0].text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=8)

# Boxplot
df_comp = df[cols].apply(lambda x: pd.to_numeric(x, errors='coerce'))
df_comp.boxplot(ax=axes[1], vert=False, patch_artist=True,
                boxprops=dict(facecolor='#97BC62', alpha=0.7),
                medianprops=dict(color='#2C5F2D', linewidth=2))
axes[1].set_title('Distribución por Competencia (Boxplot)', fontweight='bold')
axes[1].set_xlabel('Escala 1–5')

plt.suptitle('Análisis de Competencias — Graduados USTA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nCompetencia más alta : {medias.idxmax()} ({medias.max():.2f})')
print(f'Competencia más baja : {medias.idxmin()} ({medias.min():.2f})')
print(f'Media global         : {medias.mean():.2f}')

## 6. Hallazgo Clave — Segunda Lengua

In [ ]:
if 'segunda_lengua' in df.columns:
    col = pd.to_numeric(df['segunda_lengua'], errors='coerce')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Distribución de respuestas
    counts = col.value_counts().sort_index()
    colores_semaforo = ['#F96167', '#F96167', '#F9E795', '#97BC62', '#2C5F2D']
    counts.plot(kind='bar', ax=axes[0], color=colores_semaforo, edgecolor='white')
    axes[0].set_title('Distribución — Segunda Lengua', fontweight='bold')
    axes[0].set_xlabel('Puntuación (1=Muy bajo, 5=Excelente)')
    axes[0].set_ylabel('Número de graduados')
    axes[0].tick_params(axis='x', rotation=0)
    for i, v in enumerate(counts):
        axes[0].text(i, v + 5, f'{v/len(col.dropna())*100:.1f}%', ha='center', fontsize=9)
    
    # Comparación con otras competencias
    comp_seleccion = ['segunda_lengua', 'insercion_laboral', 'herramientas_modernas',
                      'trabajo_equipo', 'toma_decisiones', 'etica']
    comp_seleccion = [c for c in comp_seleccion if c in df.columns]
    medias_sel = df[comp_seleccion].apply(lambda x: pd.to_numeric(x, errors='coerce')).mean()
    colores_comp = ['#F96167' if c in ['segunda_lengua', 'insercion_laboral', 'herramientas_modernas'] else '#97BC62' for c in medias_sel.index]
    medias_sel.plot(kind='bar', ax=axes[1], color=colores_comp, edgecolor='white')
    axes[1].set_title('Brechas vs Fortalezas (selección)', fontweight='bold')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Media (1–5)')
    axes[1].set_ylim(0, 5.5)
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].axhline(y=3.0, color='red', linestyle=':', alpha=0.5, label='Umbral 3.0')
    axes[1].legend()
    for i, v in enumerate(medias_sel):
        axes[1].text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=9)
    
    plt.suptitle('Hallazgo Clave: Segunda Lengua — Brecha Transversal', fontsize=13, fontweight='bold', color='#F96167')
    plt.tight_layout()
    plt.show()
    
    pct_bajo = (col <= 2).sum() / col.notna().sum() * 100
    print(f'⚠️  El {pct_bajo:.1f}% de graduados calificó segunda lengua con 1 o 2')
    print(f'   Media global segunda lengua: {col.mean():.2f} / 5.0')

## 7. Matriz de Correlación entre Competencias

In [ ]:
cols = [c for c in COLS_COMPETENCIAS if c in df.columns]
df_comp = df[cols].apply(lambda x: pd.to_numeric(x, errors='coerce'))
corr = df_comp.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='RdYlGn', vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    square=True, linewidths=0.5
)
ax.set_title('Matriz de Correlación — Competencias de Graduados USTA', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Correlaciones más fuertes
corr_upper = corr.where(mask == False)
np.fill_diagonal(corr_upper.values, np.nan)
corr_vals = corr_upper.stack().sort_values(ascending=False)
print('Top 5 correlaciones más altas:')
for (c1, c2), v in corr_vals.head(5).items():
    print(f'  {c1} ↔ {c2}: {v:.3f}')
print(f'\nCorrelación media entre competencias: {corr_vals.mean():.3f}')

## 8. Bienestar y Movilidad Social

In [ ]:
COLS_BIENESTAR = [
    'adquirio_bienes', 'mejoro_vivienda', 'mejoro_salud',
    'acceso_seguridad_social', 'incremento_cultural',
    'satisfecho_ocio', 'red_amigos'
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Porcentaje de Sí por indicador de bienestar
cols_bien = [c for c in COLS_BIENESTAR if c in df.columns]
bien_pct = df[cols_bien].apply(lambda x: pd.to_numeric(x, errors='coerce')).mean() * 100
bien_pct = bien_pct.sort_values()
colores_bien = ['#F96167' if v < 50 else '#97BC62' for v in bien_pct]
bien_pct.plot(kind='barh', ax=axes[0], color=colores_bien, edgecolor='white')
axes[0].set_title('Indicadores de Bienestar (% que respondió Sí)', fontweight='bold')
axes[0].set_xlabel('Porcentaje (%)')
axes[0].axvline(x=50, color='gray', linestyle='--', alpha=0.7, label='50%')
axes[0].legend()
for i, v in enumerate(bien_pct):
    axes[0].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

# Movilidad social
if 'movilidad_social' in df.columns:
    mov = pd.to_numeric(df['movilidad_social'], errors='coerce').value_counts().sort_index()
    colores_mov = ['#F96167' if v < 0 else '#F9E795' if v == 0 else '#97BC62' for v in mov.index]
    mov.plot(kind='bar', ax=axes[1], color=colores_mov, edgecolor='white')
    axes[1].set_title('Índice de Movilidad Social\n(Estrato actual − Estrato al graduarse)', fontweight='bold')
    axes[1].set_xlabel('Cambio de estrato')
    axes[1].set_ylabel('Número de graduados')
    axes[1].tick_params(axis='x', rotation=0)
    mov_num = pd.to_numeric(df['movilidad_social'], errors='coerce')
    pct_ascenso = (mov_num > 0).sum() / mov_num.notna().sum() * 100
    axes[1].set_xlabel(f'Cambio de estrato\n({pct_ascenso:.1f}% ascendió de estrato)')
    for i, v in enumerate(mov):
        axes[1].text(i, v + 5, str(v), ha='center', fontsize=9)

plt.suptitle('Bienestar y Movilidad Social — Impacto de la Formación USTA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Satisfacción General

In [ ]:
COLS_SAT = {
    'satisfaccion_vida': 'Satisfacción con la vida',
    'satisfaccion_formacion': 'Satisfacción con la formación',
    'efecto_calidad_vida': 'Efecto en calidad de vida',
}

cols_sat = {k: v for k, v in COLS_SAT.items() if k in df.columns}

if cols_sat:
    fig, axes = plt.subplots(1, len(cols_sat), figsize=(16, 5))
    if len(cols_sat) == 1:
        axes = [axes]
    
    for ax, (col, titulo) in zip(axes, cols_sat.items()):
        datos = pd.to_numeric(df[col], errors='coerce').value_counts().sort_index()
        colores_sat = ['#F96167', '#F96167', '#F9E795', '#97BC62', '#2C5F2D', '#1a3a1a']
        datos.plot(kind='bar', ax=ax, color=colores_sat[:len(datos)], edgecolor='white')
        ax.set_title(titulo, fontweight='bold')
        ax.set_xlabel('Puntuación')
        ax.set_ylabel('Graduados')
        ax.tick_params(axis='x', rotation=0)
        media = pd.to_numeric(df[col], errors='coerce').mean()
        ax.set_xlabel(f'Puntuación (Media: {media:.2f})')
        for i, v in enumerate(datos):
            ax.text(i, v + 3, f'{v/len(df)*100:.1f}%', ha='center', fontsize=8)
    
    plt.suptitle('Niveles de Satisfacción — Graduados USTA', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 10. Resumen del EDA

In [ ]:
print('=' * 60)
print('  RESUMEN DEL ANÁLISIS EXPLORATORIO — PENSER USTA')
print('=' * 60)

cols_comp = [c for c in COLS_COMPETENCIAS if c in df.columns]
medias_comp = df[cols_comp].apply(lambda x: pd.to_numeric(x, errors='coerce')).mean()

print(f'''
📦 BASE DE DATOS
   Registros analizados : {len(df):,}
   Variables analíticas : {df.shape[1]}

📊 COMPETENCIAS
   Media global         : {medias_comp.mean():.2f} / 5.0
   Más alta             : {medias_comp.idxmax()} ({medias_comp.max():.2f})
   Más baja             : {medias_comp.idxmin()} ({medias_comp.min():.2f})
   Correlación media    : Alta (>0.5 entre la mayoría)

🏠 BIENESTAR
   Score promedio       : {pd.to_numeric(df.get('score_bienestar', pd.Series()), errors='coerce').mean():.2f} / 7
   Movilidad social     : ~21% ascendió de estrato

⚠️  HALLAZGOS CLAVE
   1. Segunda lengua es la brecha más crítica (media 2.76)
   2. Inserción laboral y herramientas modernas también son débiles
   3. Alta correlación entre competencias → justifica reducción dimensional
   4. El 49% alcanza perfil de Profesional Consolidado
''')
print('=' * 60)
print('  → Continúa en 02_modeling.ipynb para los arquetipos')
print('=' * 60)